# <font color=blue> __Single-cell RNA-seq data analysis using SCANPY__  </font>

- 이 주피터 노트북은 단일세포 RNA-seq 데이터를 분석하기 위한 기초 실습을 포함합니다.
- SCANPY를 이용한 전처리, 시각화를 위한 기본적인 처리 과정에 대한 코드 예제를 제공합니다.
- Google Colab에서 실습하는 것을 가정하고 만들었으나 Anaconda 등이 설치된 개인 컴퓨터에서도 사용할 수 있습니다.
<br>

- __AnnData Format:__ https://anndata.readthedocs.io/en/stable/
- __SCANPY:__ https://scanpy.readthedocs.io/en/stable/
<br>

__MLBI-lab, July 21, 2025__

## __1. Install required packages and import them__
이제 파일이 준비되었으니 필요한 패키지를 설치하고 이를 불러와서(import해서) 데이터를 들여다 볼 수 있습니다.

In [ ]:
## 필요한 패키지 설치 (이미 설치되었다면 skip 해도 됩니다.)
'''
!pip install gdown
!pip install numpy
!pip install pandas
!pip install scipy
!pip install matplotlib
!pip install scikit-learn
!pip install seaborn
!pip install scikit-network
'''
!pip install leidenalg
!pip install scanpy
!pip install scoda-viz==0.4.31b
## This jupyter notebook was tested with the above version of scoda-viz package.

In [ ]:
## 필요한 패키지 불러오기
import copy, warnings, os
import anndata
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
from scodaviz import load_sample_data, plot_sankey, show_tree

warnings.filterwarnings('ignore')

## __Python의 데이터 구조들__

### __A. (가장 중요한) 데이터 프레임__
- 데이터 프레임이란? A: 엑셀 sheet 같은 것
- index (행이름)과 columns (열이름) 확인하기
- 데이터 프레임의 정렬, 최대값, 합/평균
- 조건설정과 boolean indexing

In [ ]:
## Example Data Frame 불러와서 변수로 저장
df = sns.load_dataset('iris')

In [ ]:
## Data Frame 확인하기
df

In [ ]:
## index와 columns 확인하기

In [ ]:
## 최대값, 평균, 합계 구하기

In [ ]:
## 특정 항목(열) 기준으로 정렬하기

In [ ]:
## 조건을 설정하고 조건에 맞는 행들만 가져오기

In [ ]:
## 파일로 저장하고 저장했던 파일 불러오기

### __B. Python의 다른 데이터 구조들__
- <font color='blue'>string (문자열)</font>
- list
- <font color='blue'>dictionary</font>
- tuple
- set
- numpy.array


In [ ]:
## String 나누기, 붙이기 등

In [ ]:
## list 정의하기, 항목 추가하기, 항목 가져오기

In [ ]:
## dictionary 정의하기, 항목 추가하기와 가져오기, key와 value 보기

In [ ]:
## numpy.array (list와 달리 모든 항목이 동일한 데이터 형)

list와 dictionary의 각 항목들은 데이터 타입이 달라도 되며 list나 또 다른 dictionary, 혹은 Data Frame을 값으로 가질 수 있습니다.

In [ ]:
## dictionary에 다양한 항목들 저장해 보기

In [ ]:
## show_tree 함수로 dictionary 구조 보기
dct = {'A': 3, 'B': [1,2,3], 'DF': df, '또다른 dict': {1: df['species'], '2': df.columns.values} }

In [ ]:
show_tree(dct)

## __2A. Download a sample data__

In [ ]:
adata = load_sample_data( 'BC' )
adata

## __2A. Or, Load your h5ad file__

In [ ]:
file_h5ad = 'BRCA_GSE161529_33K_v1.h5ad'

In [ ]:
adata = sc.read_h5ad(file_h5ad)
adata

## __3. Check your data__

#### __AnnData Format:__ https://anndata.readthedocs.io/en/stable/
  
<div>
<img src="https://drive.google.com/uc?export=view&id=1IGjZpO_DeaNFiX3u_fy_rnPC1oKwVsqU" align="center" width="500"/>
</div>
<div>
<img src="attachment:fefeae20-ad05-4cea-8327-72f3dd19903f.png" width="500"/>
</div>


In [ ]:
adata

In [ ]:
adata.obs.head()

In [ ]:
adata.var.head()

In [ ]:
adata.obs[['condition']].value_counts()

In [ ]:
adata.obs[['sample']].value_counts()

In [ ]:
lst = [adata.obs['condition'], adata.obs['sample']]
plot_sankey(lst, title=None, fs=12, WH=(400, 600), th=30, title_y=0.85 )

## __4. Filtering cells and genes__
#### __Get statistics for Cell/Gene filtering__

In [ ]:
adata

In [ ]:
show_tree( adata )

In [ ]:
## select genes of which the name start with 'MT-', i.e., mitochondrial genes
adata.var['mt'] = adata.var_names.str.startswith('MT-')
## Boolean values saved to the new column named, 'mt' in var

In [ ]:
## Check the number of mitochondrial genes
display(adata.var['mt'].sum())

In [ ]:
adata

In [ ]:
## Get stats
sc.pp.calculate_qc_metrics(adata, qc_vars = ['mt'], percent_top = None, \
                           log1p = False, inplace = True)

adata
## Results of this function are stored in new columns in the obs field

pp.calculate_qc_metrics: https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.calculate_qc_metrics.html

In [ ]:
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

In [ ]:
adata.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].min()

#### __Cell/Gene filtering__
- cell-by-gene count matrix에서 각 gene이 발현된 cell 수가 너무 적은 것은 제거
- 이를 위해 cell-by-gene count matrix를 data frame으로 가져와서
- 각 gene에 대해 0보다 큰 cell의 수를 계산하여
- 이 수를 기준으로 조건을 만들어서 filtering 함

#### Easy way to cell/gene filtering

In [ ]:
sc.pp.filter_cells(adata, min_genes = 200)
sc.pp.filter_cells(adata, max_genes = 4000)
sc.pp.filter_genes(adata, min_cells = 20)

adata

In [ ]:
adata.write('BC_filtered.h5ad')

In [ ]:
b = adata.obs['pct_counts_mt'] <= 15
adata = adata[b, :]
adata

## __5. Basic processings (Clustering & Visualization)__

#### __2차원 시각화(UMAP 또는 tSNE)를 위한 몇 가지 routine 한 step__
1. Count normalization: 각 행의 합(각 cell에 대한 RNA count의 총합)을 동일한 값(예 10,000)으로 맞춤
2. 1-augmented log-transform: Raw count의 확률분포를 정규분포에 가깝게 변환 (log변환 전 1을 더함)
3. highly variable gene selection: cell마다 값들의 변동이 큰 gene들을 확인하여 var field의 열('variable_gene')에 저장
4. Dimension reduction using PCA: 선형변환 방식인 PCA를 이용하여 각 cell의 expression pattern(X의 각 행)을 차원 축소 -> obsm['X_pca']에 저장됨. (_Note:_ PCA를 위한 차원은 소위 'explained variance' plot을 보고 결정하면 좋지만 대략 15정도로 설정해도 무방함.)
5. Building Neighbor graph: PCA로 차원 축소한 벡터를 이용하여, 각 cell에 대해 expression pattern이 유사한 k개의 cell을 선택하여 이를 sparse matrix 형태로 저장 (matrix 크기는 cell수xcell수 이며 obsp field에 sparse matrix로 저장 됨.
6. UMAP or tSNE projection: Neighbor graph까지 얻어졌으면 umap 또는 tsne를 이용하여 2차원으로 차원 축소하여 산점도 확인 가능
7. Clustering: 비슷한 expression pattern을 갖는 cell들을 하나의 클러스터로 그룹핑

In [ ]:
## 1. Count normalization
sc.pp.normalize_total(adata, target_sum=1e4)
adata

In [ ]:
## 2. 1-augmented log-transform
sc.pp.log1p(adata)
adata

In [ ]:
adata.uns['log1p']

In [ ]:
## 3. highly variable gene selection
sc.pp.highly_variable_genes(adata, n_top_genes = 2000)
adata

In [ ]:
adata.var['highly_variable'].sum()

In [ ]:
## check stats of highly-variable genes
sc.pl.highly_variable_genes(adata)

In [ ]:
## 4. Dimension reduction using PCA
sc.tl.pca(adata, n_comps = 15, use_highly_variable = True)
adata

In [ ]:
adata.uns['pca']

In [ ]:
adata.obsm['X_pca'].shape

In [ ]:
## 5. Building Neighbor graph
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=15, use_rep = 'X_pca')
adata

In [ ]:
adata.obsp['connectivities']

In [ ]:
## 6a: Compute UMAP projection
sc.tl.umap(adata)
adata

In [ ]:
## 6b: tSNE projection (It takes time)
sc.tl.tsne(adata)
adata

In [ ]:
adata.obsm['X_umap']

In [ ]:
## 7: Clustering using leiden algorithm
sc.tl.leiden(adata, resolution = 1, key_added = 'cluster')
## The clustering resolution was intentionally set low for quick example. Default value is 1.
adata

In [ ]:
adata.obs['cluster'].value_counts()

## __6. Visualization__

#### 이제 UMAP 또는 tSNE 좌표를 이용해 대체적인 분포를 시각화해 볼 수 있습니다.

In [ ]:
items_to_plot = ['condition', 'cluster', 'sample']

plt.rcParams['figure.figsize'] = (3.5, 3.5)
sc.pl.embedding( adata, color = items_to_plot,
                 basis = 'X_umap', neighbors_key = 'neighbors',
                 wspace = 0.4, hspace = 0.3, legend_fontsize = 10,
                 ncols = 2, palette = 'Spectral', add_outline = True)

In [ ]:
items_to_plot = ['condition', 'cluster', 'sample']

plt.rcParams['figure.figsize'] = (3.5, 3.5)
sc.pl.embedding( adata, color = items_to_plot,
                 basis = 'X_tsne', neighbors_key = 'neighbors',
                 wspace = 0.4, hspace = 0.3, legend_fontsize = 10,
                 ncols = 2, palette = 'Spectral', add_outline = True)

#### 잘 알려진 몇가지 marker gene들의 발현량을 보면 어느 cluster가 무슨 세포유형인지 대략적으로 확인 가능

In [ ]:
genes_to_plot = ['CD3D', 'CD4', 'CD8A',     ## T cell markers
                 'CD79A', 'MS4A1', 'MZB1',  ## B cell markers
                 'CD14', 'LYZ',             ## Myeloid cell markers
                 'FBLN1', 'NOTCH3',         ## Stromal cells (Fibroblast, Smooth muscle cell) markers
                 'EPCAM', 'MUC1',           ## Epithelial cell markers
                 'CD34']                    ## Endothelial cell marker

plt.rcParams['figure.figsize'] = (3.5, 3.5)
sc.pl.embedding( adata, color = genes_to_plot + ['cluster'], use_raw = False,
                 basis = 'X_umap', neighbors_key = 'neighbors',
                 wspace = 0.2, hspace = 0.3, legend_fontsize = 8,
                 ncols = 3, palette = 'Spectral', add_outline = True, vmax = 'p99', vmin = 0)

#### Cluster 별 manual cell-type annotation을 할 때 요런 plot을 먼저 봅니다.

In [ ]:
genes_to_plot_dct = { 'T': ['CD3D', 'CD4', 'CD8A'],     ## T cell markers
                      'B': ['CD79A', 'MS4A1', 'MZB1'],  ## B cell markers
                      'M': ['CD14', 'LYZ'],             ## Myeloid cell markers
                      'S': ['FBLN1', 'NOTCH3'],         ## Stromal cells (Fibroblast, Smooth muscle cell) markers
                      'Epi': ['EPCAM', 'MUC1'],         ## Epithelial cell markers
                      'Endo': ['CD34'] }                ## Endothelial cell marker

In [ ]:
sc.pl.stacked_violin(adata, genes_to_plot_dct, groupby='cluster', figsize = (6,6))

In [ ]:
adata.obs['my_celltype_annot'] = 'NA'
b = adata.obs['cluster'].isin( ['0', '11', '16'] )
adata.obs.loc[b, 'my_celltype_annot'] = 'M'

b = adata.obs['cluster'].isin( ['1', '2', '23', '25'] )
adata.obs.loc[b, 'my_celltype_annot'] = 'T'

In [ ]:
plt.rcParams['figure.figsize'] = (3.5, 3.5)
sc.pl.embedding( adata, color = items_to_plot + ['my_celltype_annot', 'CD3D', 'LYZ'],
                 basis = 'X_tsne', neighbors_key = 'neighbors',
                 wspace = 0.4, hspace = 0.3, legend_fontsize = 10,
                 ncols = 2, palette = 'Spectral', add_outline = True)

In [ ]:
sc.pl.dotplot(adata, genes_to_plot_dct, groupby='cluster', figsize = (6,6))

In [ ]:
fig, axes = plt.subplots(len(genes_to_plot), 1, figsize=(8, 2*len(items_to_plot)), dpi = 80)
plt.subplots_adjust(hspace = 0.9)

for j, item in enumerate(genes_to_plot):
    sc.pl.violin(adata, [item], groupby='cluster', ax = axes[j], use_raw = False, stripplot = False, show = False)

In [ ]:
genes_to_plot_dct = {'T': ['CD3D', 'CD4', 'CD8A'],     ## T cell markers
                 'B': ['CD79A', 'MS4A1', 'MZB1'],  ## B cell markers
                 'M': ['CD14', 'LYZ'],             ## Myeloid cell markers
                 'S': ['FBLN1', 'NOTCH3'],         ## Stromal cells (Fibroblast, Smooth muscle cell) markers
                 'Epi': ['EPCAM', 'MUC1'],           ## Epithelial cell markers
                 'Endo': ['CD34'] }                   ## Endothelial cell marker


sc.pl.heatmap( adata, var_names = genes_to_plot_dct, groupby = 'cluster',
               var_group_labels = genes_to_plot_dct.keys(),
               standard_scale = True, figsize = (6, 8) )

## __7. DEG Analysis (among clusters)__

In [ ]:
adata

In [ ]:
test_sel = 't-test'  ## 'logreg', 't-test', 'wilcoxon', 't-test_overestim_var'
sc.tl.rank_genes_groups(adata, groupby = 'cluster', method=test_sel,
                        key_added = test_sel, pts = True)
adata

In [ ]:
adata.uns['t-test'].keys()

#### Get DEG results in data frame

In [ ]:
target_cluster = '1'
gene_rank = sc.get.rank_genes_groups_df(adata, group = target_cluster,
                                        key = test_sel, pval_cutoff = 0.01,
                                        log2fc_min = 0.25)
gene_rank.iloc[:30]

In [ ]:
gene_rank.shape

In [ ]:
target_cluster = '18'
gene_rank = sc.get.rank_genes_groups_df(adata, group = target_cluster,
                                        key = test_sel, pval_cutoff = 0.01,
                                        log2fc_min = 0.25)
gene_rank.iloc[:30]

#### Show the DEG results

In [ ]:
sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False,
                        key = test_sel, fontsize = 6)